In [ ]:
!pip install pandas sentence-transformers scikit-learn matplotlib

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report
import warnings

# Tắt các cảnh báo không cần thiết
warnings.filterwarnings("ignore")

# 1. CẤU HÌNH DANH SÁCH EMBEDDING MODELS
# Đã loại bỏ CodeBERT và thêm 3 model theo yêu cầu
embedding_models_list = [
    'all-MiniLM-L6-v2',                                # Giữ lại làm chuẩn (Baseline)
    'sentence-transformers/all-mpnet-base-v2',         # Tôi khuyến nghị dùng v2 thay vì v1, nhưng bạn có thể sửa lại thành v1 nếu muốn
    'sentence-transformers/sentence-t5-base',          # Lưu ý: Model này chạy sẽ lâu hơn chút và cần thư viện sentencepiece
    'sentence-transformers/paraphrase-TinyBERT-L6-v2'  # Model nhẹ, chạy nhanh
]

In [ ]:


# 2. LOAD DỮ LIỆU
file_path = 'solidity_with_minilm_embeddings-part2.xlsx'

try:
    df = pd.read_csv(file_path)
except:
    # Fallback nếu file tên gốc chưa đổi đuôi
    df = pd.read_excel('solidity_with_minilm_embeddings-part2.xlsx')

print(f"-> Đã tải dữ liệu: {len(df)} dòng.")
print(f"-> Các cột: {list(df.columns)}")

# Lấy input và label
texts = df['functional_semantics'].astype(str).tolist()
labels_raw = df['label'].tolist()

# Mã hóa nhãn (String -> Number)
le = LabelEncoder()
y = le.fit_transform(labels_raw)
print(f"-> Các nhãn (Classes): {le.classes_}")

# 3. HÀM ĐÁNH GIÁ (LOOP)
results = {}

def evaluate_model(model_name):
    print(f"\n{'='*20}\nĐang xử lý model: {model_name}")

    # Load Model
    try:
        model = SentenceTransformer(model_name)
    except Exception as e:
        print(f"Lỗi tải model {model_name}: {e}")
        return 0

    # Embedding
    print("1. Đang tạo Embedding (vui lòng chờ)...")
    X = model.encode(texts, show_progress_bar=True)

    # Chia tập dữ liệu
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

    # Cấu hình MLP (Tăng max_iter để đảm bảo hội tụ)
    mlp = MLPClassifier(hidden_layer_sizes=(256, 128), activation='relu', solver='adam',
                        max_iter=1000, random_state=42, early_stopping=True)

    # Train
    print("2. Đang huấn luyện MLP...")
    mlp.fit(X_train, y_train)

    # Đánh giá
    y_pred = mlp.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    print(f"-> Độ chính xác (Accuracy): {acc*100:.2f}%")

    return acc

# 4. CHẠY THỰC NGHIỆM
for model_name in embedding_models_list:
    acc = evaluate_model(model_name)
    results[model_name] = acc

# 5. VẼ BIỂU ĐỒ SO SÁNH
print("\n================ TỔNG KẾT KẾT QUẢ ================")
# Sắp xếp kết quả từ cao xuống thấp
sorted_results = dict(sorted(results.items(), key=lambda item: item[1], reverse=True))

for model, acc in sorted_results.items():
    print(f"{model:<45} | {acc*100:.2f}%")

# Vẽ biểu đồ cột
plt.figure(figsize=(12, 6))
bars = plt.barh(list(sorted_results.keys()), [v*100 for v in sorted_results.values()], color='teal')
plt.xlabel('Độ chính xác (%)')
plt.title('So sánh hiệu quả các Embedding Models trên bài toán phân loại')
plt.xlim(0, 100)

# Thêm số liệu lên cột
for bar in bars:
    width = bar.get_width()
    plt.text(width + 1, bar.get_y() + bar.get_height()/2, f'{width:.1f}%', ha='left', va='center')

plt.tight_layout()
plt.show()

In [ ]:
!pip install torch torch-geometric pandas numpy solc-select

In [ ]:
import os
import re
import subprocess
import json
import torch
import pandas as pd
import numpy as np
import torch.nn.functional as F
from torch.nn import Linear, Embedding, Sequential, ReLU, BatchNorm1d
from torch_geometric.data import Data, DataLoader
from torch_geometric.nn import GATv2Conv, GCNConv, SAGEConv, GINConv, global_mean_pool
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report
import matplotlib.pyplot as plt

# ================= CONFIGURATION =================
INPUT_EXCEL = "solidity_code_with_labels-3k-full.xlsx" # Tên file data của bạn
ETHERSOLVE_JAR = "EtherSolve.jar"  # Đường dẫn đến file jar
OUTPUT_DIR = "processed_graphs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Thiết lập thiết bị (GPU nếu có)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# ================= PART 1: CFG EXTRACTION UTILS =================
def extract_version(code):
    match = re.search(r"pragma\s+solidity\s+\^?([0-9]+\.[0-9]+(?:\.[0-9]+)?)", code)
    return match.group(1) if match else "0.4.24" # Default fallback

def extract_contract_name(code):
    match = re.search(r'\bcontract\s+([A-Za-z_][A-Za-z0-9_]*)\b', code)
    return match.group(1) if match else "Contract"

def run_shell(cmd):
    subprocess.run(cmd, shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# Hàm chuyển Opcode thành Vector Index
opcode_vocab = {"UNK": 0}
def get_opcode_id(op):
    if op not in opcode_vocab:
        opcode_vocab[op] = len(opcode_vocab)
    return opcode_vocab[op]

def process_single_contract(row_idx, code, label):
    """
    Biên dịch code -> Lấy CFG -> Chuyển thành PyTorch Geometric Data
    """
    try:
        version = extract_version(code)
        name = extract_contract_name(code)

        # 1. Setup Solidity Version
        run_shell(f"solc-select install {version}")
        run_shell(f"solc-select use {version}")

        # 2. Compile to Bin-Runtime
        with open("temp.sol", "w", encoding="utf-8") as f:
            f.write(code)

        # Compile (có thể thất bại nếu code lỗi, ta dùng try-catch)
        ret = subprocess.run(f"solc --bin-runtime temp.sol -o output --overwrite",
                             shell=True, capture_output=True)

        bin_path = f"output/{name}.bin-runtime"
        if not os.path.exists(bin_path):
            return None # Compile lỗi

        run_shell(f"cp {bin_path} temp.evm")

        # 3. Gen CFG bằng EtherSolve
        run_shell(f"java -jar {ETHERSOLVE_JAR} -r -j -o report.json temp.evm")

        if not os.path.exists("report.json"):
            return None

        # 4. Parse JSON to Graph Data
        with open("report.json") as f:
            cfg = json.load(f)

        nodes = cfg["runtimeCfg"]["nodes"]
        edges_raw = cfg["runtimeCfg"]["successors"]

        if not nodes: return None

        # Xử lý Node Features (Opcode Embedding Indices)
        # Chúng ta lấy 20 opcode đầu tiên của mỗi Block làm đặc trưng
        MAX_OP_LEN = 20
        node_feats = []

        for node in nodes:
            parsed = node.get("parsedOpcodes", "")
            ops = [line.split(":")[1].strip().split()[0] for line in parsed.split("\n") if ":" in line]

            # Padding hoặc Truncate
            op_ids = [get_opcode_id(op) for op in ops]
            if len(op_ids) < MAX_OP_LEN:
                op_ids += [0] * (MAX_OP_LEN - len(op_ids))
            else:
                op_ids = op_ids[:MAX_OP_LEN]

            node_feats.append(op_ids)

        x = torch.tensor(node_feats, dtype=torch.long) # [Num_Nodes, Feature_Dim]

        # Xử lý Edges
        offset_map = {n["offset"]: i for i, n in enumerate(nodes)}
        edge_indices = []
        for e in edges_raw:
            src = offset_map.get(e["from"])
            for dst_offset in e["to"]:
                dst = offset_map.get(dst_offset)
                if src is not None and dst is not None:
                    edge_indices.append([src, dst])

        if not edge_indices:
            edge_index = torch.empty((2, 0), dtype=torch.long)
        else:
            edge_index = torch.tensor(edge_indices, dtype=torch.long).t().contiguous()

        return Data(x=x, edge_index=edge_index, y=torch.tensor([label], dtype=torch.long))

    except Exception as e:
        return None

# ================= PART 2: DEFINING GNN MODELS =================

# 1. GATv2 (Theo yêu cầu)
class GATv2Model(torch.nn.Module):
    def __init__(self, vocab_size, hidden_dim, num_classes):
        super().__init__()
        self.embedding = Embedding(vocab_size, hidden_dim)
        # GATv2Conv layers
        self.conv1 = GATv2Conv(hidden_dim, hidden_dim, heads=4, concat=True)
        self.conv2 = GATv2Conv(hidden_dim * 4, hidden_dim, heads=1, concat=False)
        self.classifier = Linear(hidden_dim, num_classes)

    def forward(self, x, edge_index, batch):
        # x: [num_nodes, seq_len] -> Embed -> Mean -> [num_nodes, hidden]
        x = self.embedding(x).mean(dim=1)
        x = self.conv1(x, edge_index).relu()
        x = self.conv2(x, edge_index)
        x = global_mean_pool(x, batch) # Pooling toàn đồ thị
        return self.classifier(x)

# 2. GCN (Graph Convolutional Network) - Baseline chuẩn
class GCNModel(torch.nn.Module):
    def __init__(self, vocab_size, hidden_dim, num_classes):
        super().__init__()
        self.embedding = Embedding(vocab_size, hidden_dim)
        self.conv1 = GCNConv(hidden_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.classifier = Linear(hidden_dim, num_classes)

    def forward(self, x, edge_index, batch):
        x = self.embedding(x).mean(dim=1)
        x = self.conv1(x, edge_index).relu()
        x = self.conv2(x, edge_index).relu()
        x = global_mean_pool(x, batch)
        return self.classifier(x)

# 3. GraphSAGE (Mạnh về tổng hợp thông tin hàng xóm)
class GraphSAGEModel(torch.nn.Module):
    def __init__(self, vocab_size, hidden_dim, num_classes):
        super().__init__()
        self.embedding = Embedding(vocab_size, hidden_dim)
        self.conv1 = SAGEConv(hidden_dim, hidden_dim)
        self.conv2 = SAGEConv(hidden_dim, hidden_dim)
        self.classifier = Linear(hidden_dim, num_classes)

    def forward(self, x, edge_index, batch):
        x = self.embedding(x).mean(dim=1)
        x = self.conv1(x, edge_index).relu()
        x = self.conv2(x, edge_index).relu()
        x = global_mean_pool(x, batch)
        return self.classifier(x)

# 4. GIN (Graph Isomorphism Network) - Mạnh nhất về phân loại đồ thị
class GINModel(torch.nn.Module):
    def __init__(self, vocab_size, hidden_dim, num_classes):
        super().__init__()
        self.embedding = Embedding(vocab_size, hidden_dim)

        # GIN cần một mạng MLP bên trong nó
        nn1 = Sequential(Linear(hidden_dim, hidden_dim), ReLU(), Linear(hidden_dim, hidden_dim))
        self.conv1 = GINConv(nn1)
        self.bn1 = BatchNorm1d(hidden_dim)

        nn2 = Sequential(Linear(hidden_dim, hidden_dim), ReLU(), Linear(hidden_dim, hidden_dim))
        self.conv2 = GINConv(nn2)
        self.bn2 = BatchNorm1d(hidden_dim)

        self.classifier = Linear(hidden_dim, num_classes)

    def forward(self, x, edge_index, batch):
        x = self.embedding(x).mean(dim=1)
        x = F.relu(self.conv1(x, edge_index))
        x = self.bn1(x)
        x = F.relu(self.conv2(x, edge_index))
        x = self.bn2(x)
        x = global_mean_pool(x, batch)
        return self.classifier(x)

# ================= PART 3: MAIN EXECUTION =================
if __name__ == "__main__":
    print("🚀 Bắt đầu quá trình xử lý...")

    # --- BƯỚC 1: LOAD VÀ PREPROCESS DATA ---
    # Nếu đã chạy rồi thì có thể load file cache, ở đây mình làm từ đầu
    try:
        df = pd.read_excel(INPUT_EXCEL)
    except:
        df = pd.read_csv(INPUT_EXCEL.replace('.xlsx', '.csv')) # Fallback csv

    # Mã hóa nhãn
    le = LabelEncoder()
    df['label_idx'] = le.fit_transform(df['label'].astype(str))
    num_classes = len(le.classes_)
    print(f"Classes found: {le.classes_}")

    dataset = []
    print("⏳ Đang trích xuất CFG và tạo Dataset (có thể lâu)...")

    # Chỉ chạy demo 100 mẫu nếu muốn test nhanh, bỏ [:100] để chạy full
    for idx, row in df.iterrows():
        if pd.isna(row['code']): continue

        graph_data = process_single_contract(idx, row['code'], row['label_idx'])

        if graph_data:
            dataset.append(graph_data)

        if (idx+1) % 50 == 0:
            print(f"Processed {idx+1}/{len(df)} files...")

    print(f"✅ Đã tạo xong {len(dataset)} đồ thị hợp lệ.")
    print(f"Vocab Size (Số lượng Opcode độc nhất): {len(opcode_vocab)}")

    # Chia Train/Test
    train_data, test_data = train_test_split(dataset, test_size=0.2, random_state=42)
    train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
    test_loader = DataLoader(test_data, batch_size=32, shuffle=False)

    # --- BƯỚC 2: TRAINING LOOP VÀ SO SÁNH ---

    vocab_size = len(opcode_vocab) + 1
    hidden_dim = 64

    models_to_test = {
        "GATv2": GATv2Model(vocab_size, hidden_dim, num_classes),
        "GCN": GCNModel(vocab_size, hidden_dim, num_classes),
        "GraphSAGE": GraphSAGEModel(vocab_size, hidden_dim, num_classes),
        "GIN": GINModel(vocab_size, hidden_dim, num_classes)
    }

    results = {}

    for name, model in models_to_test.items():
        print(f"\nTraining Model: {name}...")
        model = model.to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=0.005)
        criterion = torch.nn.CrossEntropyLoss()

        # Train loop
        model.train()
        for epoch in range(20): # Train 20 epochs
            total_loss = 0
            for batch in train_loader:
                batch = batch.to(device)
                optimizer.zero_grad()
                out = model(batch.x, batch.edge_index, batch.batch)
                loss = criterion(out, batch.y)
                loss.backward()
                optimizer.step()
                total_loss += loss.item()

            # Print loss every 5 epochs
            if (epoch+1) % 5 == 0:
                print(f"Epoch {epoch+1}: Loss {total_loss/len(train_loader):.4f}")

        # Eval loop
        model.eval()
        all_preds = []
        all_labels = []
        with torch.no_grad():
            for batch in test_loader:
                batch = batch.to(device)
                out = model(batch.x, batch.edge_index, batch.batch)
                pred = out.argmax(dim=1)
                all_preds.extend(pred.cpu().numpy())
                all_labels.extend(batch.y.cpu().numpy())

        acc = accuracy_score(all_labels, all_preds)
        results[name] = acc
        print(f"-> {name} Accuracy: {acc*100:.2f}%")

    # --- BƯỚC 3: VẼ BIỂU ĐỒ SO SÁNH ---
    print("\n================ FINAL COMPARISON ================")
    sorted_results = dict(sorted(results.items(), key=lambda item: item[1], reverse=True))

    plt.figure(figsize=(10, 6))
    bars = plt.bar(sorted_results.keys(), [v*100 for v in sorted_results.values()], color=['#FF5733', '#33FF57', '#3357FF', '#FF33A1'])
    plt.ylabel('Accuracy (%)')
    plt.title('Comparison of Graph Embedding Models on Solidity CFGs')
    plt.ylim(0, 100)

    for bar in bars:
        yval = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2, yval + 1, f"{yval:.1f}%", ha='center', va='bottom')

    plt.savefig("graph_model_comparison.png")
    plt.show()
    print("Biểu đồ đã được lưu vào 'graph_model_comparison.png'")